# 06 — BM'nin doğurganlık varsayımı ve tarihsel deneyim (Görsel 9)

**Soru:** BM, bugün doğurganlığı 1,5'in altında olan ülkeler için toparlanma öngörüyor. Geçmişte 1,5'in altına düşen ülkeler gerçekte ne yaptı?

**Yöntem:** 161 büyük ülke. Tarihsel kısım yalnızca 1950–2023 (BM tahminleri). Karşılaştırma aynı uzunlukta iki dönem üzerinden yapılır: geçmişte bir ülkenin 1,5'in altına **ilk düştüğü yıldan sonraki 27 yıl** ile BM'nin bugün 1,5'in altındaki ülkeler için öngördüğü **2023 → 2050 (27 yıl)**.

**Sınırlılıklar:** Tarihsel örnek küçük (27 ülke) ve büyük kısmı 1990'larda Sovyet sonrası krizle 1,5'in altına düşen Doğu Avrupa ülkeleri; bugünkü düşüşün nedenleri farklı olabilir. BM'nin modeli de bu tarihsel deneyimle kalibre ediliyor. Bulgu "BM yanılıyor" değil, "BM'nin varsayımı tarihten daha tekdüze bir toparlanma öngörüyor".

**Çıktı:** `data/out/g09b_dogurganlik_toparlanma.csv` (grafik), `data/out/g09b_toparlanma_ozet.csv` (ülke özeti).

In [1]:
import pandas as pd, numpy as np

veri = pd.read_csv("../data/clean/countries_clean.csv")
ulkeler = pd.read_csv("../data/clean/ulkeler_1m.csv")
ESIK = 1.5

d = veri[veri.ISO3_code.isin(ulkeler.iso3) & (veri.Time <= 2100)][["ISO3_code", "Location", "Time", "TFR"]]
p = d.pivot(index="ISO3_code", columns="Time", values="TFR")
isim = d.drop_duplicates("ISO3_code").set_index("ISO3_code").Location
gecmis = p.loc[:, 1950:2023]
print(p.shape)

(161, 151)


## 1. Tarih: 1,5'in altına düşen ülkeler sonra ne yaptı?

In [2]:
satir = []
for iso, x in gecmis.iterrows():
    alt = x[x < ESIK]
    if alt.empty:
        continue
    t0 = int(alt.index.min())
    sonrasi = x.loc[t0:]
    t_min = int(sonrasi.idxmin())
    satir.append({"iso3": iso, "country": isim[iso], "first_below_year": t0,
                  "min_tfr": round(sonrasi.min(), 3), "min_year": t_min,
                  "max_after_min": round(x.loc[t_min:].max(), 3), "tfr_2023": round(x.loc[2023], 3),
                  "change_27y": round(x.loc[t0 + 27] - x.loc[t0], 3) if t0 + 27 <= 2023 else np.nan})
tarih = pd.DataFrame(satir)

print("1950–2023 arasında 1,5'in altına düşen ülke:", len(tarih))
for e in [1.7, 1.8, 2.1]:
    print(f"  Dip sonrası {e} ve üstüne çıkan: {(tarih.max_after_min >= e).sum()}")
print("  2023'te hâlâ 1,5'in altında:", (tarih.tfr_2023 < ESIK).sum(),
      "| 2023'te 1,8 ve üstünde:", (tarih.tfr_2023 >= 1.8).sum())

1950–2023 arasında 1,5'in altına düşen ülke: 51
  Dip sonrası 1.7 ve üstüne çıkan: 11
  Dip sonrası 1.8 ve üstüne çıkan: 3
  Dip sonrası 2.1 ve üstüne çıkan: 0
  2023'te hâlâ 1,5'in altında: 41 | 2023'te 1,8 ve üstünde: 0


## 2. Aynı uzunlukta karşılaştırma: tarih ve BM

In [3]:
gec = tarih.dropna(subset=["change_27y"])          # 27 yıllık takibi olan ülkeler
bugun = p[p[2023] < ESIK]                          # bugün 1,5'in altındakiler
bm27 = bugun[2050] - bugun[2023]

karsilastirma = pd.DataFrame({
    "Tarih (ilk düşüşten sonraki 27 yıl)": [len(gec), gec.change_27y.mean(), gec.change_27y.median(), (gec.change_27y > 0).mean() * 100],
    "BM (2023 → 2050)": [len(bugun), bm27.mean(), bm27.median(), (bm27 > 0).mean() * 100]},
    index=["Ülke sayısı", "Ortalama TFR değişimi", "Medyan TFR değişimi", "Artan ülke payı (%)"]).round(2)
display(karsilastirma)
print("BM 2100: artan pay", round(((bugun[2100] - bugun[2023]) > 0).mean() * 100), "% | 1,5 ve üstüne çıkan pay",
      round((bugun[2100] >= ESIK).mean() * 100), "%")

,Tarih (ilk düşüşten sonraki 27 yıl),BM (2023 → 2050)
Ülke sayısı,27.00,41.00
Ortalama TFR değişimi,0.01,0.11
Medyan TFR değişimi,0.01,0.10
Artan ülke payı (%),51.85,100.00


BM 2100: artan pay 100 % | 1,5 ve üstüne çıkan pay 46 %


## 3. Türkiye: son 10 yıl ve BM'nin sonraki 10 yılı

In [4]:
tr = p.loc["TUR"]
print("Türkiye TFR:", {y: round(tr[y], 2) for y in [2000, 2013, 2023, 2033, 2050, 2100]})
print("Son 10 yıl:", round(tr[2023] - tr[2013], 2), "| BM'ye göre sonraki 10 yıl:", round(tr[2033] - tr[2023], 2))

Türkiye TFR: {2000: np.float64(2.49), 2013: np.float64(2.11), 2023: np.float64(1.62), 2033: np.float64(1.61), 2050: np.float64(1.62), 2100: np.float64(1.61)}
Son 10 yıl: -0.48 | BM'ye göre sonraki 10 yıl: -0.01


## 4. Çıktılar

In [5]:
grup = d[d.ISO3_code.isin(tarih.iso3)].copy()
grup["is_projection"] = grup.Time > 2023
grup["below_1_5_in_2023"] = grup.ISO3_code.isin(bugun.index)
grup["recovered_to_1_8"] = grup.ISO3_code.isin(tarih[tarih.max_after_min >= 1.8].iso3)
grup.columns = ["iso3", "country", "year", "tfr", "is_projection", "below_1_5_in_2023", "recovered_to_1_8"]
grup.to_csv("../data/out/g09b_dogurganlik_toparlanma.csv", index=False)
tarih.to_csv("../data/out/g09b_toparlanma_ozet.csv", index=False)
print("Kaydedildi:", grup.shape, tarih.shape)

Kaydedildi: (7701, 7) (51, 8)
